# Step 5: Profit levers for the board

This notebook turns Steps 2–4 into board decisions with dollar figures, ranked by size. It adds one new lever, branch pricing, and an experiment design for the 10% discount.

**How to read the dollar figures.** Every dollar figure is a **conditional annualised gross-profit opportunity**: H1 gross-profit difference × 365/181, *if* the stated condition holds. They are **not** net profit (no labour, rent or overheads), **not** promised savings, and they **must not be added together** into one forecast: each depends on a different condition, and they interact (for example, Garden's price lever depends on which cost table is right).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw = pd.read_csv(ROOT / "data" / "raw" / "stories_transactions.csv")
raw["Date"] = pd.to_datetime(raw["Date"])
ANN = 365 / 181
DAYS = raw["Date"].nunique()
price = raw.groupby(["Branch", "Product"])["Unit Price (USD)"].first().unstack(0)
cost = raw.groupby(["Branch", "Product"])["Unit Cost (USD)"].first().unstack(0)
assert (raw.groupby(["Branch", "Product"])[["Unit Price (USD)", "Unit Cost (USD)"]].nunique() == 1).all().all()
print(f"{len(raw):,} lines, {DAYS} days; prices and costs are fixed per branch and product over the period")
print("Price ratio vs Harbour, all 45 products (min-max):",
      {b: f"{(price[b]/price['Harbour']).min():.3f}-{(price[b]/price['Harbour']).max():.3f}" for b in ["Garden", "Uptown"]})
print("Cost ratio vs Harbour (min-max):",
      {b: f"{(cost[b]/cost['Harbour']).min():.3f}-{(cost[b]/cost['Harbour']).max():.3f}" for b in ["Garden", "Uptown"]})

51,177 lines, 181 days; prices and costs are fixed per branch and product over the period
Price ratio vs Harbour, all 45 products (min-max): {'Garden': '0.951-0.953', 'Uptown': '0.912-0.916'}
Cost ratio vs Harbour (min-max): {'Garden': '1.426-1.439', 'Uptown': '0.952-0.965'}


## 1. Garden cost audit (largest lever; detail in Step 3)

In [2]:
g = raw[raw.Branch == "Garden"].copy()
g["harbour_cost"] = g.Product.map(cost["Harbour"])
gap = ((g["Unit Cost (USD)"] - g["harbour_cost"]) * g.Quantity).sum()
print(f"Fact: Garden recorded-cost difference vs Harbour unit costs, H1: ${gap:,.0f}")
print(f"Conditional annualised gross-profit opportunity: ${gap*ANN:,.0f}/yr "
      "(only if Garden really pays more than Harbour AND equivalent supply terms are feasible)")

Fact: Garden recorded-cost difference vs Harbour unit costs, H1: $15,923
Conditional annualised gross-profit opportunity: $32,111/yr (only if Garden really pays more than Harbour AND equivalent supply terms are feasible)


## 2. Branch pricing: what a price change must survive

**Facts.** On all 45 products, Garden prices are about 95.2% of Harbour's and Uptown prices about 91.4%. Prices never change in the data, so **the data cannot show how customers would react to a price change**. Similar hours or basket sizes across branches do not show similar willingness to pay.

**What we can calculate** is the break-even: how much volume a branch could lose before a price rise reduces gross profit. If volume falls by less than that, gross profit rises.

Break-even volume loss = 1 − (current gross profit / gross profit at new prices with unchanged volume). **Volume means units sold**, with the product mix and discount rates kept as recorded.

Garden is shown under **recorded costs** and under **normalised costs** (Harbour's unit costs). Its break-even depends heavily on which cost table is right, which is another reason to do the cost audit first.

In [3]:
def price_scenario(branch, step, cost_basis):
    x = raw[raw.Branch == branch]
    unit_cost = x["Unit Cost (USD)"] if cost_basis == "recorded" else x.Product.map(cost["Harbour"])
    new_price = x["Unit Price (USD)"] + step * (x.Product.map(price["Harbour"]) - x["Unit Price (USD)"])
    rev0 = x["Line Revenue (USD)"].sum()
    rev1 = (x.Quantity * new_price * (1 - x["Discount Rate"])).sum()
    gp0 = rev0 - (x.Quantity * unit_cost).sum()
    gp1 = gp0 + (rev1 - rev0)
    return {"branch": branch, "cost basis": cost_basis,
            "price change": f"{'half' if step == 0.5 else 'full'} gap to Harbour (+{rev1/rev0-1:.1%})",
            "H1 gross profit now": round(gp0), "margin now": f"{gp0/rev0:.1%}",
            "opportunity if volume unchanged ($/yr)": round((rev1 - rev0) * ANN),
            "break-even volume loss": f"{1 - gp0/gp1:.1%}"}

pricing = pd.DataFrame([price_scenario(b, s, c) for b, c in [("Uptown", "recorded"), ("Garden", "recorded"), ("Garden", "normalised (Harbour costs)")]
                        for s in [0.5, 1.0]])
pricing

,branch,cost basis,price change,H1 gross profit now,margin now,opportunity if volume unchanged ($/yr),break-even volume loss
0,Uptown,recorded,half gap to Harbour (+4.7%),34882,62.6%,5272,7.0%
1,Uptown,recorded,full gap to Harbour (+9.4%),34882,62.6%,10545,13.0%
2,Garden,recorded,half gap to Harbour (+2.5%),46794,46.9%,5026,5.1%
3,Garden,recorded,full gap to Harbour (+5.0%),46794,46.9%,10053,9.6%
4,Garden,normalised (Harbour costs),half gap to Harbour (+2.5%),62718,62.9%,5026,3.8%
5,Garden,normalised (Harbour costs),full gap to Harbour (+5.0%),62718,62.9%,10053,7.4%


**Reading.**
* **Uptown** (costs about 96% of Harbour's, so no cost question): closing the full price gap is +9.4%. It is a gross-profit opportunity of about $10.5k/yr **if volume is unchanged**, and it stays positive unless Uptown loses about 13% or more of its volume.
* **Garden**: the break-even is *higher* under recorded costs (9.6% for the full gap) than under normalised costs (7.4%). With a thin recorded margin, each lost sale costs less recorded profit, so a price rise looks safer than it may be. The opportunity in dollars is the same under both cost bases, because costs do not change with price. Because Garden's cost table is in question (Step 3) and Garden is the second-largest branch, **we recommend the pricing experiment at Uptown first**.

### Uptown half-gap (+4.7%) downside scenarios

**Scenario, not a forecast.** Annualised GP change = ((1 − assumed unit loss) × GP at new prices with unchanged units − current GP) × 365/181. Product mix, unit costs and discount rates unchanged; H1 annualised; **implementation costs excluded**. This is a full-year run-rate; first-year savings after a pilot would be smaller (the pilot period and set-up come first).

In [4]:
x = raw[raw.Branch == "Uptown"]
new_price = x["Unit Price (USD)"] + 0.5 * (x.Product.map(price["Harbour"]) - x["Unit Price (USD)"])
gp_now = x["Line Gross Profit (USD)"].sum()
gp_new = gp_now + (x.Quantity * new_price * (1 - x["Discount Rate"])).sum() - x["Line Revenue (USD)"].sum()
downside = pd.DataFrame({"assumed unit loss": ["0%", "3%", "5%", "7%", "10%"],
                         "annualised GP change ($/yr)": [round(((1 - L) * gp_new - gp_now) * ANN) for L in [0, .03, .05, .07, .10]]})
downside

,assumed unit loss,annualised GP change ($/yr)
0,0%,5272
1,3%,3004
2,5%,1491
3,7%,-21
4,10%,-2289


At a 5% unit loss, about $1,491/yr of gross-profit benefit remains to cover incremental implementation and ongoing costs (price updates, menus, staff time). If those costs are higher, the pilot is not worth running at that demand loss.

## 3. Uptown pricing experiment: design and stop/review rules

* **Change:** raise Uptown prices by **half the gap** to Harbour first (about +4.7%), not the full gap, to limit downside.
* **Primary business outcome:** Uptown **gross profit** per day, compared with the pre-pilot weeks. Detecting a demand loss is not the same as proving that the price rise increases profit.
* **Guardrail metric:** Uptown's share of chain **units sold** (Uptown units ÷ units at all three branches, same days). The share is used because it adjusts for chain-wide growth and the Tuesday pattern.
* **Share is not the same as units.** If *s* is Uptown's initial share and *d* its proportional unit decline, and the other branches are unchanged, the relative share decline is *r* = *d*(1 − *s*)/(1 − *s·d*); conversely *d* = *r*/(1 − *s* + *s·r*). The conversion assumes stable comparison branches, unchanged product mix and unchanged costs. **Branch switching** (Uptown customers moving to Garden or Harbour) or unrelated changes at the comparison branches invalidate this simple conversion.
* **Proposed business guardrail (half-gap pilot):** the half-gap break-even unit loss (≈ 6.97%) corresponds to a relative share decline of ≈ 5.73% (computed below). Two consecutive weekly breaches of that share decline can trigger reversal to the original prices. This is a business rule, **not a statistical test and not a guarantee against losses**: single weeks are noisy, so it can trigger on noise or miss a real loss.
* **Test length:** 8 weeks is an **initial pilot that can remain inconclusive**. The detectable-change estimate below is a rough planning approximation only; it does not guarantee power and does not control for all confounding.
* **Unclear result:** restore the original pre-pilot prices.

In [5]:
daily = raw.groupby(["Date", "Branch"])["Quantity"].sum().unstack()   # units sold, matching the break-even
share = daily["Uptown"] / daily.sum(axis=1)
weekly = share.groupby(share.index.to_period("W")).mean().iloc[1:-1]   # drop partial first/last weeks
sd_w = weekly.diff().dropna().std() / np.sqrt(2)                       # week-to-week noise, trend removed by differencing
base = weekly.mean()
for weeks in [4, 8]:
    # minimum detectable change in mean weekly share (pre vs post, equal weeks, 80% power, 5% two-sided)
    mde = 2.8 * sd_w * np.sqrt(2 / weeks)
    r = mde / base
    d = r / (1 - base + base * r)
    print(f"{weeks} weeks before vs {weeks} after: detectable relative change in Uptown's unit share ≈ {r:.2%} "
          f"≈ {d:.2%} unit decline if comparison branches are unchanged — rough planning approximation")
be_half = float(pricing.iloc[0]["break-even volume loss"].rstrip("%")) / 100
be_exact = 1 - pricing.iloc[0]["H1 gross profit now"] / (pricing.iloc[0]["H1 gross profit now"] + pricing.iloc[0]["opportunity if volume unchanged ($/yr)"] / ANN)
print(f"Uptown initial unit share s = {base:.2%}")
print(f"Half-gap break-even unit loss {be_exact:.2%} -> relative share decline {be_exact*(1-base)/(1-base*be_exact):.2%} (proposed guardrail)")
print(f"A 13% relative share decline -> {0.13/(1-base+base*0.13):.2%} unit decline")

4 weeks before vs 4 after: detectable relative change in Uptown's unit share ≈ 10.12% ≈ 12.20% unit decline if comparison branches are unchanged — rough planning approximation
8 weeks before vs 8 after: detectable relative change in Uptown's unit share ≈ 7.16% ≈ 8.69% unit decline if comparison branches are unchanged — rough planning approximation
Uptown initial unit share s = 18.96%
Half-gap break-even unit loss 6.97% -> relative share decline 5.73% (proposed guardrail)
A 13% relative share decline -> 15.57% unit decline


This is a planning estimate under the assumption that weekly noise stays as in H1. The 8-week detectable change (≈ 7.2% of share, ≈ 8.7% of units) is **larger** than the half-gap break-even (≈ 6.97% of units). So an 8-week pilot can miss a loss large enough to wipe out the gain and can remain inconclusive. In that case, restore the original prices.

## 4. 10% discount: test removal, don't just remove it

**Facts.** The 10% discount appears on about 5.7% of lines at every branch, time and product. It cost $1,707 in H1. The data does not show whether it brings in sales.

In [6]:
d10 = raw[raw["Discount Rate"] == 0.10]
given = (d10.Quantity * d10["Unit Price (USD)"] * 0.10).sum()
print(f"Fact: 10% discount given in H1: ${given:,.0f} (${given/DAYS:.2f}/day chain-wide)")
print(f"Conditional annualised gross-profit opportunity: ${given*ANN:,.0f}/yr, only if removing it loses no sales")

gp_day = raw.groupby("Date")["Line Gross Profit (USD)"].sum()
X = pd.get_dummies(gp_day.index.day_name()).astype(float).values
X = np.column_stack([X, np.arange(len(gp_day))])
resid = gp_day.values - X @ np.linalg.lstsq(X, gp_day.values, rcond=None)[0]
sd = resid.std(ddof=X.shape[1])
for n in [14, 28, 56]:
    mde = 2.8 * sd * np.sqrt(2 / n)
    print(f"{n} discount days vs {n} no-discount days (randomly assigned within weeks): "
          f"detectable gross-profit change ≈ ${mde:,.0f}/day vs ${given/DAYS:.2f}/day at stake — Planning estimate")

Fact: 10% discount given in H1: $1,707 ($9.43/day chain-wide)
Conditional annualised gross-profit opportunity: $3,443/yr, only if removing it loses no sales
14 discount days vs 14 no-discount days (randomly assigned within weeks): detectable gross-profit change ≈ $98/day vs $9.43/day at stake — Planning estimate
28 discount days vs 28 no-discount days (randomly assigned within weeks): detectable gross-profit change ≈ $69/day vs $9.43/day at stake — Planning estimate
56 discount days vs 56 no-discount days (randomly assigned within weeks): detectable gross-profit change ≈ $49/day vs $9.43/day at stake — Planning estimate


**Reading.** The discount is worth about $9/day chain-wide. Normal day-to-day variation in gross profit is far larger, so even a two-month test **cannot reliably detect** an effect of that size. The test is therefore a **risk check for clear problems**, not a measurement of the effect:

* **Design:** remove the 10% discount on randomly chosen days, balanced within each week, at all branches.
* **Weekly review (operational):** check for customer complaints, staff or POS problems, and a gross-profit shortfall on no-discount days larger than a tolerance the board sets in advance as a business choice. The detectable-change figures above are **not** used as stopping thresholds; they only show how little a test of this length can resolve.
* **End-of-test reading:** not finding significant harm does **not** show that removing the discount is profitable. With an effect this small, the most likely result is "inconclusive".
* **Decision after the test:** the board chooses to keep the discount, remove it as a business judgement accepting the uncertainty, or continue testing. The $3.4k/yr remains a conditional opportunity, not a demonstrated saving.

## 5. Board decision table

In [7]:
up = pricing[(pricing.branch == "Uptown")]
board = pd.DataFrame([
    {"rank": 1, "decision": "Audit Garden's cost table before any margin-based decision about Garden",
     "conditional annualised GP opportunity": f"up to ${gap*ANN:,.0f}/yr",
     "condition": "Garden really pays more than Harbour and equivalent terms are feasible",
     "evidence": "Fact: ÷0.7 rule on 45/45 products (Step 3)",
     "owner": "Finance / procurement",
     "first action": "Pull supplier invoices for bottled water and 3–5 high-volume products at Garden and Harbour; check the costing rule in the POS/ERP",
     "implementation cost considerations": "Staff time only; renegotiating or switching suppliers may involve contract, logistics and quality costs"},
    {"rank": 2, "decision": "Run an Uptown pricing experiment (half the gap to Harbour first)",
     "conditional annualised GP opportunity": f"${up.iloc[0]['opportunity if volume unchanged ($/yr)']:,}/yr (half gap) to ${up.iloc[1]['opportunity if volume unchanged ($/yr)']:,}/yr (full gap)",
     "condition": f"units sold unchanged (same mix); positive while unit loss < {up.iloc[0]['break-even volume loss']} (half) / {up.iloc[1]['break-even volume loss']} (full)",
     "evidence": "Fact: prices ≈91.4% of Harbour's; demand response unknown",
     "owner": "Commercial / pricing lead with the Uptown branch manager",
     "first action": "Fix the stop rule and test length; record pre-change weekly unit share",
     "implementation cost considerations": "POS price update, menu and signage reprint, staff briefing; risk to customer goodwill and of customers switching branches"},
    {"rank": 3, "decision": "Test removing the 10% discount as a risk-checked experiment",
     "conditional annualised GP opportunity": f"up to ${given*ANN:,.0f}/yr",
     "condition": "removal loses no sales; a short test cannot confirm this",
     "evidence": "Fact: $1,707 in H1, no targeting pattern",
     "owner": "Operations with finance",
     "first action": "Find out what triggers the 10% discount at the till (policy, staff discretion, POS setting)",
     "implementation cost considerations": "POS configuration to switch the discount by day, staff training, possible customer complaints"},
    {"rank": 4, "decision": "Do not repeat Harbour's 15% campaign unchanged; test with randomised offer days",
     "conditional annualised GP opportunity": "avoids an estimated −$308 / −$391 GP per campaign",
     "condition": "model assumptions in Step 2",
     "evidence": "Model estimate (Step 2)",
     "owner": "Marketing with the Harbour branch manager",
     "first action": "Pause repeat campaigns until a randomised-day test is designed",
     "implementation cost considerations": "Planning time; offer days must be followed exactly at the till"},
    {"rank": 5, "decision": "Validate customer-history targeting before committing marketing budget",
     "conditional annualised GP opportunity": "no dollar figure",
     "condition": "—",
     "evidence": "No demonstrated advantage over random selection (Step 4)",
     "owner": "Marketing with IT / POS",
     "first action": "Ask the POS team how Customer ID is attached to a purchase",
     "implementation cost considerations": "Audit time; a verified-ID pilot needs POS or app changes and staff time at the till"},
])
board

,rank,decision,conditional annualised GP opportunity,condition,evidence,owner,first action,implementation cost considerations
0,1,Audit Garden's cost table before any margin-ba...,"up to $32,111/yr",Garden really pays more than Harbour and equiv...,Fact: ÷0.7 rule on 45/45 products (Step 3),Finance / procurement,Pull supplier invoices for bottled water and 3...,Staff time only; renegotiating or switching su...
1,2,Run an Uptown pricing experiment (half the gap...,"$5,272/yr (half gap) to $10,545/yr (full gap)",units sold unchanged (same mix); positive whil...,Fact: prices ≈91.4% of Harbour's; demand respo...,Commercial / pricing lead with the Uptown bran...,Fix the stop rule and test length; record pre-...,"POS price update, menu and signage reprint, st..."
2,3,Test removing the 10% discount as a risk-check...,"up to $3,443/yr",removal loses no sales; a short test cannot co...,"Fact: $1,707 in H1, no targeting pattern",Operations with finance,Find out what triggers the 10% discount at the...,POS configuration to switch the discount by da...
3,4,Do not repeat Harbour's 15% campaign unchanged...,avoids an estimated −$308 / −$391 GP per campaign,model assumptions in Step 2,Model estimate (Step 2),Marketing with the Harbour branch manager,Pause repeat campaigns until a randomised-day ...,Planning time; offer days must be followed exa...
4,5,Validate customer-history targeting before com...,no dollar figure,—,No demonstrated advantage over random selectio...,Marketing with IT / POS,Ask the POS team how Customer ID is attached t...,Audit time; a verified-ID pilot needs POS or a...
